# Kestrel V3, step 2: DPO

Starts from step 1's model and pushes it toward preferred answers. `Run all` does whichever
stage is next, and every stage resumes.

| stage | runtime | what | cost |
|---|---|---|---|
| A | **CPU** | trainer self-test, build the pairs (`scripts/prepare_dpo.py`), save to Drive | ~0.1 unit |
| B | **A100** | DPO from `runs/sft_v3` (`scripts/train_dpo.py`), capped at ~1.2 hours | ~8 units |
| C | **A100** | the chat eval on the DPO model, and recount the earlier transcripts with the fixed counters | ~1.5 units |

Stage A stops the notebook when it's done. Switch the runtime to A100 and `Run all` again.


In [ ]:
# 0. Which runtime?
import os, json, shutil, glob, subprocess
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('torch', torch.__version__, '|', GPU or 'CPU runtime')
if GPU and 'A100' not in GPU and 'H100' not in GPU:
    raise SystemExit(f'Got {GPU}. Use a CPU runtime for stage A, or an A100 for stages B and C.')


In [ ]:
# 1. Drive. The first mount right after the permission prompt usually fails with
#    'mount failed', and a second attempt works with no prompt.
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError as e:
    print('first mount failed, retrying:', e)
    drive.mount('/content/drive', force_remount=True)
KD = '/content/drive/MyDrive/Kestrel'
assert os.path.isdir(KD), f'{KD} not found - mounted the wrong Google account?'
DPO = f'{KD}/datasets/dpo_v3'
RUN = f'{KD}/runs/dpo_v3'
SFT = f'{KD}/runs/sft_v3/ckpt.pt'
print('step 1 checkpoint:', os.path.exists(SFT), '| pairs on Drive:', os.path.exists(f'{DPO}/manifest.json'))


In [ ]:
# 2. Code (public repo)
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip -q install tokenizers datasets


In [ ]:
# A. Self-test and pairs (CPU runtime). manifest.json is written last.
if not GPU:
    r = subprocess.run(['python', '-m', 'scripts.train_dpo', '--self-test'], capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-1500:])
    assert 'SELF_TEST_OK' in r.stdout, 'self-test failed - see above'
if not os.path.exists(f'{DPO}/manifest.json'):
    if GPU:
        raise SystemExit('The pairs are not on Drive yet. Stage A is CPU work: switch to a CPU runtime.')
    !python -u -m scripts.prepare_dpo --out /content/data_dpo
    assert os.path.exists('/content/data_dpo/manifest.json'), 'build failed - see the log above'
    os.makedirs(DPO, exist_ok=True)
    for f in ['tokens.bin', 'index.npy', 'manifest.json']:
        shutil.copy(f'/content/data_dpo/{f}', f'{DPO}/{f}')
    drive.flush_and_unmount()
    print(open('/content/data_dpo/manifest.json').read())
if not GPU:
    raise SystemExit('STAGE_A_DONE. Switch the runtime to A100 and Run all for stages B and C.')


In [ ]:
# B. DPO. --max-steps caps the run so it can't eat step 4's budget, whatever the real speed is.
def local(src, dst):
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f'{src}/*'):
        out = f'{dst}/{os.path.basename(f)}'
        if not os.path.exists(out) or os.path.getsize(out) != os.path.getsize(f):
            shutil.copy(f, out)
local(DPO, '/content/data_dpo')
local(f'{KD}/datasets/sft_v3', '/content/data_sft_v3')
def finished():
    p = f'{RUN}/status.json'
    return os.path.exists(p) and json.load(open(p)).get('state') == 'done'
if not finished():
    !python -u -m scripts.train_dpo --ckpt {SFT} --data-dir /content/data_dpo --sft-dir /content/data_sft_v3 \
        --out {RUN} --max-steps 900
assert finished(), 'DPO did not finish - see the log above. Run all again to resume.'
print('STAGE_B_DONE')


In [ ]:
# C. Same fixed questions as V1 and step 1, then recount every transcript with the fixed counters.
OUT = f'{KD}/evals/chat_eval_v3.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
if 'step2' not in done:
    !python -u -m scripts.eval_chat --ckpt {RUN}/ckpt.pt --label step2 --out {OUT}
!python -u -m scripts.eval_chat --rescore --out {OUT}
print('STAGE_C_DONE')
